In [20]:
import pandas as pd
import numpy as np
import os

# Cargar datos
ruta = os.path.join("..", 'Data', 'TouristAccommodationRaw26012026.csv')

df = pd.read_csv(ruta, encoding='latin1')

## 1. Columnas para multiples departamentos

En esta fase, preparamos unas columnas que facilitan el analisis de varios departamentos. El objetivo principal es normalizar las categorías y asegurar que las métricas sean columnas númericas sin reducir el tamaño de la muestra (XXXX registros).

* **Índice unico** Se crea una columna `unique_id` para identificar el id unico de todos los registros.
* **Normalización de Texto:** Se eliminan espacios en blanco en las columnas `city` y `room_type` para evitar duplicidad de categorías.
* **Consistencia de Tipos:** Se fuerza el tipo a `float` en las columnas de reseñas para permitir cálculos estadísticos precisos.

In [21]:
# (1) Crear una columna de indice para identificar el id unico de registros

# Este columna de indice 'unique_id' con valor empieza desde 1 hasta el numero de todos registros
df['unique_id'] = np.arange(1, len(df)+1)

In [22]:
# (2) NORMALIZACIÓN DE CATEGORÍAS

# Eliminamos espacios en blanco, estandarizamos a minúsculas y capitalizamos
# (ej. "madrid " -> "Madrid")
df['city'] = df['city'].str.strip().str.capitalize()


# Limpiamos los tipos de alojamiento
df['room_type'] = df['room_type'].str.strip()

In [ ]:
# --- 3. IMPUTACIÓN DE PRECIOS ---

# Paso A: Imputación segmentada (Por Ciudad y Tipo de Habitación)
df['price'] = df.groupby(['city', 'room_type'])['price'].transform(
    lambda x: x.fillna(x.median())
)

# Paso B: Imputación de seguridad
# En caso de que un segmento completo sea nulo, usamos la mediana de todo el dataset
df['price'] = df['price'].fillna(df['price'].median())

# --- 4. AJUSTE DE TIPOS ---
# Aseguramos que price sea numérico para el análisis comercial
df['price'] = df['price'].astype(float)

In [ ]:
# (5) Columnas de reseñas por clientes, para analisis de Marketing y Experiencia de Clientes

# Definición de las columnas de reseñas
columnas_rating = [
    'review_scores_rating',        # Evaluación general
    'review_scores_accuracy',      # Precisión de detalles
    'review_scores_cleanliness',   # Higiene
    'review_scores_checkin',       # Proceso de entrada
    'review_scores_communication', # Comunicación
    'review_scores_location',      # Zona
    'review_scores_value'          # Valor
]

# Aseguramos el tipo numérico. Los errores o celdas vacías se convierten en NaN.
for col in columnas_rating:
    df[col] = pd.to_numeric(df[col], errors='coerce')

## 2. Limpieza y Preparación: Marketing y Estrategia Comercial

En esta fase, preparamos los datos para el análisis de mercado. El objetivo principal es normalizar las categorías y asegurar que las métricas sean columnas númericas sin reducir el tamaño de la muestra (XXXX registros).

* **Normalización de Texto:** Se crea una columna `neighbourhood` para los barrios con nombres codificados correctamente.
* **Consistencia de Tipos:** Se fuerza el tipo a `float` en las columnas `minimum_nights` y `maximum_nights` para permitir cálculos estadísticos precisos.

In [24]:
# (1) Limpieza para los nombres de los barrios

# Creamos una columna "neighbourhood" que tenga el valor de 'neighbourhood_district'
# En los casos de nullos (Mallorca, Menorca, Girona, Malaga), tenga el valor de 'neighbourhood_name'
df['neighbourhood'] = df['neighbourhood_district'].fillna(df['neighbourhood_name'])

In [25]:
# Muchos barrios tienen los nombres mal codificados en la base de datos aunque usamos "encoding =" con cualquier método

# Creamos un diccionario para guardar los nombres corrctos de los mal codificados,
# con la ayuda de IA identificando todos los valores únicos de "neighbourhood".
# Aqui los signos falsos son del metodo encoding='latin1'
cleaning_city = {
    # Barcelona
    'Sant Martï¿½': 'Sant Martí', 'Grï¿½cia': 'Gràcia', 'Sarriï¿½-Sant Gervasi': 'Sarrià-Sant Gervasi',
    'Sants-Montjuï¿½c': 'Sants-Montjuïc', 'Horta-Guinardï¿½': 'Horta-Guinardó',
    # Madrid
    'Chamartï¿½n': 'Chamartín', 'Tetuï¿½n': 'Tetuán', 'Chamberï¿½': 'Chamberí', 'Vicï¿½lvaro': 'Vicálvaro',
    # Mallorca & Menorca
    'Alcï¿½dia': 'Alcúdia', 'Sï¿½ller': 'Sóller', 'Santanyï¿½': 'Santanyí', 'Llubï¿½': 'Llubí',
    'Calviï¿½': 'Calvià', 'Pollenï¿½a': 'Pollença', 'Marratxï¿½': 'Marratxí', 'Artï¿½': 'Artà',
    'Alarï¿½': 'Alaró', 'Bï¿½ger': 'Búger', 'Deyï¿½': 'Deià', 'Santa Eugï¿½nia': 'Santa Eugènia',
    'Mahï¿½n': 'Mahón', 'Sant Lluï¿½s': 'Sant Lluís', 'Santa Marï¿½a del Camï¿½': 'Santa Maria del Camí',
    'Sant Llorenï¿½ des Cardassar': 'Sant Llorenç des Cardassar',
    'Montuï¿½ri': 'Montuïri', 
    # Sevilla & Valencia
    'Nerviï¿½n': 'Nervión', 'LA SAIDIA': 'LA SAÏDIA', 'ALGIROS': 'ALGIRÓS', 
    'POBLATS MARITIMS': 'POBLATS MARÍTIMS', 'JESUS': 'JESÚS',
    # Girona
    'Torroella de Fluviï¿½': 'Torroella de Fluvià', 'Castellï¿½ d\'Empï¿½ries': 'Castelló d\'Empúries',
    'Cadaquï¿½s': 'Cadaqués', 'Palamï¿½s': 'Palamós', 'Celrï¿½': 'Celrà', 'Llanï¿½ï¿½': 'Llançà',
    'Sant Feliu de Guï¿½xols': 'Sant Feliu de Guíxols', 'Pontï¿½s': 'Pontós', 'Tortellï¿½': 'Tortellà',
    'Cornellï¿½ del Terri': 'Cornellà del Terri', 'Lladï¿½': 'Lladó', 'Urï¿½s': 'Urús', 'Vilaï¿½r': 'Vilaür',
    'Vidrï¿½': 'Vidrà', 'Bescanï¿½': 'Bescanó', 'Serinyï¿½': 'Serinyà', 'Ventallï¿½': 'Ventalló',
    'Bellcaire d\'Empordï¿½': 'Bellcaire d\'Empordà', 'Besalï¿½': 'Besalú', 'Ullï¿½': 'Ullà',
    'Puigcerdï¿½': 'Puigcerdà', 'Sant Martï¿½ de Llï¿½mena': 'Sant Martí de Llémena', 'Foixï¿½': 'Foixà',
    'La Tallada d\'Empordï¿½': 'La Tallada d\'Empordà', 'Vallfogona de Ripollï¿½s': 'Vallfogona de Ripollès',
    'Regencï¿½s': 'Regencós', 'Sant Pau de Segï¿½ries': 'Sant Pau de Segúries', 'Maï¿½anet de la Selva': 'Maçanet de la Selva',
    'Parlavï¿½': 'Parlavà', 'Maiï¿½ de Montcal': 'Maià de Montcal', 'Bï¿½scara': 'Bàscara',
    'Arbï¿½cies': 'Arbúcies', 'Vilajuï¿½ga': 'Vilajüiga', 'Torroella de Montgrï¿½': 'Torroella de Montgrí',
    'Sant Juliï¿½ de Ramis': 'Sant Julià de Ramis', 'Palau de Santa Eulï¿½lia': 'Palau de Santa Eulàlia',
    'Sant Martï¿½ Vell': 'Sant Martí Vell', "Cruï¿½lles, Monells i Sant Sadurnï¿½ de l'Heura": "Cruïlles, Monells i Sant Sadurní de l'Heura",
    'Campdevï¿½nol': 'Campdevànol', 'Rabï¿½s': 'Rabós'
}

# Sustuimso los nombres mal codificados por los correctos
df['neighbourhood'] = df['neighbourhood'].replace(cleaning_city)


# Verificamos si todavía quedan barrios mal con nombres mal codificados
remaining_bad = df[df['neighbourhood'].str.contains('ï¿½', na=False)]['neighbourhood'].unique()
if len(remaining_bad) == 0:
    print("EXITO: Todos errores identificados de encoding se han limpiado!")
else:
    print(f"Quedan problemáticos: {remaining_bad}")

EXITO: Todos errores identificados de encoding se han limpiado!


In [26]:
# Ponemos los nombres de barrios valencianos en forma Titulo como los de otras ciudades
df.loc[df['city'] == 'Valencia', 'neighbourhood'] = df.loc[df['city'] == 'Valencia', 'neighbourhood'].str.title()

In [27]:
# (2) Columns de disponibilidad mínima y máxima de noches

# Asegurar el tipo numérico
df['minimum_nights'] = pd.to_numeric(df['minimum_nights'], errors='coerce')
df['maximum_nights'] = pd.to_numeric(df['maximum_nights'], errors='coerce')

## 3. Limpieza para Experiencia del Cliente.

Para responder a las preguntas sobre satisfacción sin reducir la muestra total de 8000 registros, se aplica la siguiente lógica:

* **Tratamiento de Ratings:** Se transforman a formato numérico, manteniendo los valores ausentes como `NaN`.

In [28]:
#Columns de cantidad de reseñas

# Asegurar el tipo numérico
df['number_of_reviews'] = pd.to_numeric(df['number_of_reviews'], errors='coerce')
df['reviews_per_month'] = pd.to_numeric(df['reviews_per_month'], errors='coerce')

## 4. Limpieza para Operaciones y Gestión de Inventario.

In [ ]:
# Asegurar el tipo numérico
df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce')
df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
df['beds'] = pd.to_numeric(df['beds'], errors='coerce')

IMPUTACIÓN DE VALORES NULOS Y CEROS

BEDROOMS
Estrategia:

Para Private room, Shared room y Hotel room → se imputa como 1 dormitorio/
Para Entire home/apt → se utiliza la mediana de dormitorios según room_type y accommodates

In [ ]:
# Referencia: solo valores válidos de bedrooms (>0)
bedrooms_ref = df[df['bedrooms'] > 0]

# Mediana de bedrooms por tipo de habitación y capacidad
bedrooms_median = bedrooms_ref.groupby(
    ['room_type', 'accommodates']
)['bedrooms'].median()

def impute_bedrooms(row):
    if pd.isna(row['bedrooms']) or row['bedrooms'] == 0:
        if row['room_type'] in ['Private room', 'Shared room', 'Hotel room']:
            return 1
        key = (row['room_type'], row['accommodates'])
        if key in bedrooms_median:
            return bedrooms_median[key]
        return 1
    return row['bedrooms']

df['bedrooms'] = df.apply(impute_bedrooms, axis=1)

BATHROOMS
Estrategia:

Se imputan valores nulos o iguales a 0 utilizando la mediana por room_type y bedrooms
En caso de no existir grupo, se usa la mediana global

In [ ]:
# Mediana de bathrooms por tipo de habitación y dormitorios
bathroom_median = df[df['bathrooms'] > 0].groupby(
    ['room_type', 'bedrooms']
)['bathrooms'].median()

def impute_bathrooms(row):
    if pd.isna(row['bathrooms']) or row['bathrooms'] == 0:
        key = (row['room_type'], row['bedrooms'])
        if key in bathroom_median:
            return bathroom_median[key]
        return df['bathrooms'].median()
    return row['bathrooms']

df['bathrooms'] = df.apply(impute_bathrooms, axis=1)

 BEDS
Estrategia:

Se utiliza la mediana por room_type y accommodates
Se asegura coherencia lógica con el número de dormitorios

In [ ]:
# Mediana de beds por tipo de habitación y capacidad
beds_median = df[df['beds'] > 0].groupby(
    ['room_type', 'accommodates']
)['beds'].median()

def impute_beds(row):
    if pd.isna(row['beds']) or row['beds'] == 0:
        key = (row['room_type'], row['accommodates'])
        if key in beds_median:
            return max(row['bedrooms'], beds_median[key])
        return max(row['bedrooms'], 1)
    return row['beds']

df['beds'] = df.apply(impute_beds, axis=1)

RESTRICCIONES LÓGICAS

In [ ]:
# Valores mínimos razonables
df['bedrooms'] = df['bedrooms'].clip(lower=1)
df['beds'] = df['beds'].clip(lower=1)
df['bathrooms'] = df['bathrooms'].clip(lower=0.5)

# Coherencia: el número de camas no puede ser menor que el número de dormitorios
df.loc[df['beds'] < df['bedrooms'], 'beds'] = df['bedrooms']

CONVERSIÓN DE TIPOS DE DATOS

In [ ]:
df['bedrooms'] = df['bedrooms'].round().astype(int)
df['beds'] = df['beds'].round().astype(int)
df['bathrooms'] = df['bathrooms'].astype(float)

In [ ]:
#Crear una columna de ocupacion_30(en dias)
df["ocupacion_30"] = 30 - df["availability_30"]

## 4. Limpieza para KPIs

In [ ]:
#COMPROBAR AVAILABILITY

df_hav = df[
    (df['availability_30'] > df['availability_60']) &
    (df['availability_60'] > df['availability_90']) &
    (df['availability_90'] > df['availability_365'])
].copy()

df_hav


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 8000 non-null   int64  
 1   name                         7997 non-null   object 
 2   description                  7946 non-null   object 
 3   host_id                      8000 non-null   int64  
 4   neighbourhood_name           8000 non-null   object 
 5   neighbourhood_district       4861 non-null   object 
 6   room_type                    8000 non-null   object 
 7   accommodates                 8000 non-null   int64  
 8   bathrooms                    7957 non-null   float64
 9   bedrooms                     7961 non-null   float64
 10  beds                         7992 non-null   float64
 11  amenities_list               7983 non-null   object 
 12  price                        7829 non-null   float64
 13  minimum_nights    

## 6. Exportación de Resultados
Una vez finalizado el proceso de limpieza y normalización para los tres departamentos se procede a exportar el Dataset Limpio.

* **Ruta de destino:** Se almacena en la carpeta institucional /Data/ bajo el nombre TouristAccommodationClean19012026.csv.
* **Codificación:** Se utiliza `latin1` para garantizar que la corrección de caracteres especiales.

In [ ]:
# --- EXPORTACIÓN DEL DATASET LIMPIO ---

# 1. Nombre del nuevo archivo
nombre_archivo_limpio = 'TouristAccommodationClean02022026.csv'

# 2. Construimos la ruta apuntando a la misma carpeta 'Data'
# Usamos '..' para subir un nivel y luego entrar en 'Data'
ruta_guardado = os.path.join('..', 'Data', nombre_archivo_limpio)

# 3. Guardamos el DataFrame
# index=False evita que se cree una columna extra de números
# encoding='latin1' 
df.to_csv(ruta_guardado, index=False, encoding='latin1')